# 00 — Data Preprocessing (Content-Based)

Notebook 1/4 of the Content-Based pipeline.

**Output files** (all with `cb_` prefix):
- `cb_books.parquet` — processed book catalog
- `cb_train_interactions.parquet` — training set
- `cb_test_interactions.parquet` — test set

**Data source:** `goodreads_interactions_dedup.json.gz` (JSONL gzip)

**Constraints:** 10 GB RAM max. All heavy ops are chunked/streamed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import gzip, json, os, gc, math, psutil
from tqdm import tqdm

def print_ram(label=''):
    mem = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    tag = f' [{label}]' if label else ''
    print(f'  [RAM{tag}] {mem:.2f} GB')

# ============================================================
# CONFIG
# ============================================================
DATA_DIR   = '/content/drive/My Drive/Thesis/Data'
OUTPUT_DIR = '/content/drive/My Drive/Thesis/book_recsys/data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BOOKS_RAW      = os.path.join(DATA_DIR, 'goodreads_books.json')
AUTHORS_FILE   = os.path.join(DATA_DIR, 'goodreads_book_authors.json.gz')
GENRES_FILE    = os.path.join(DATA_DIR, 'goodreads_book_genres_initial.json.gz')
INTERACT_RAW   = os.path.join(DATA_DIR, 'goodreads_interactions_dedup.json.gz')

BOOKS_OUT      = os.path.join(OUTPUT_DIR, 'cb_books.parquet')
PARTITION_DIR  = os.path.join(OUTPUT_DIR, 'cb_partitions')
VALID_USERS    = os.path.join(OUTPUT_DIR, 'cb_valid_users.parquet')
TRAIN_OUT      = os.path.join(OUTPUT_DIR, 'cb_train_interactions.parquet')
TEST_OUT       = os.path.join(OUTPUT_DIR, 'cb_test_interactions.parquet')

os.makedirs(PARTITION_DIR, exist_ok=True)

HEX_CHARS      = '0123456789abcdef'
JSONL_CHUNK    = 1_000_000   # lines per processing chunk
BOOK_FLUSH     = 50_000      # books per write batch
MIN_INTERACT   = 5           # minimum interactions per user
TEST_RATIO     = 0.2

print('Config done.')
print_ram()

Config done.
  [RAM] 0.62 GB


## 1. Process Books

Stream `goodreads_books.json` (all books, no limit). Lookup author names and genres. Write `cb_books.parquet`.

In [ ]:
if os.path.exists(BOOKS_OUT):
    print(f'SKIP: {os.path.basename(BOOKS_OUT)} already exists.')
    print_ram()
else:
    # --- Load author dict ---
    print('Loading authors...')
    author_map = {}
    with gzip.open(AUTHORS_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f):
            r = json.loads(line)
            author_map[r['author_id']] = r['name']
    print(f'  Authors: {len(author_map):,}')
    print_ram('after authors')

    # --- Load genre dict ---
    print('Loading genres...')
    genre_map = {}
    with gzip.open(GENRES_FILE, 'rt', encoding='utf-8') as f:
        for line in tqdm(f):
            r = json.loads(line)
            gs = list(r.get('genres', {}).keys())
            if gs:
                genre_map[r['book_id']] = gs
    print(f'  Genres: {len(genre_map):,} books')
    print_ram('after genres')

    # --- Stream books ---
    def _main_author(authors_list):
        if not authors_list:
            return 'Unknown'
        return author_map.get(authors_list[0].get('author_id'), 'Unknown')

    print('Processing ALL books (streaming)...')
    writer = None
    buf = []
    total = 0

    with open(BOOKS_RAW, 'r', encoding='utf-8') as f:
        for line in tqdm(f):
            data = json.loads(line)
            if not data.get('title'):
                continue
            b_id = data['book_id']
            title = data.get('title_without_series') or data['title']
            shelves_raw = data.get('popular_shelves', [])
            top_shelves = [s['name'] for s in shelves_raw[:15]]
            buf.append({
                'book_id': b_id,
                'title': title,
                'author': _main_author(data.get('authors', [])),
                'description': data.get('description', ''),
                'shelves': ', '.join(top_shelves),
                'genres': ', '.join(genre_map.get(b_id, [])),
                'average_rating': data.get('average_rating', '0.0'),
                'ratings_count': str(data.get('ratings_count', '0')),
                'image_url': data.get('image_url', ''),
                'language_code': data.get('language_code', ''),
                'num_pages': str(data.get('num_pages', '')),
                'publication_year': str(data.get('publication_year', '')),
            })
            total += 1
            if len(buf) >= BOOK_FLUSH:
                tbl = pa.Table.from_pandas(pd.DataFrame(buf), preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(BOOKS_OUT, tbl.schema)
                writer.write_table(tbl)
                print(f'  Written {total:,} books', end='\r')
                buf.clear()
                del tbl
                gc.collect()

    if buf:
        tbl = pa.Table.from_pandas(pd.DataFrame(buf), preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(BOOKS_OUT, tbl.schema)
        writer.write_table(tbl)
        buf.clear()
    if writer:
        writer.close()

    print(f'\nTotal books: {total:,}')
    del author_map, genre_map
    gc.collect()
    print_ram('after books')

SKIP: cb_books.parquet already exists.
  [RAM] 0.62 GB


## 2. Interactions — Step 1: Stream JSONL + Hash Partitioning

Read `goodreads_interactions_dedup.json.gz` line-by-line (JSONL gzip).
- Filter: `rating == 0` removed, books not in catalog removed
- Parse `date_added` to **Unix timestamp (int64)**
- Hash-partition by first hex char of `user_id` into 16 Parquet files

**Chunk strategy:** Accumulate 1M valid lines into a buffer, then flush to 16 partition writers. This keeps RAM under control.

In [ ]:
# Check if partitions already exist
_parts_exist = all(
    os.path.exists(os.path.join(PARTITION_DIR, f'part_{h}.parquet'))
    for h in HEX_CHARS
)

if _parts_exist:
    print('SKIP: All 16 partition files already exist.')
    print_ram()
else:
    # Load valid book_id set from cb_books.parquet
    print('Loading valid book_ids from cb_books.parquet...')
    valid_books = set(
        pq.read_table(BOOKS_OUT, columns=['book_id']).column('book_id').to_pylist()
    )
    print(f'  Valid books: {len(valid_books):,}')
    print_ram('after loading book ids')

    # Define Parquet schema for partitions
    part_schema = pa.schema([
        ('user_id',   pa.string()),
        ('book_id',   pa.string()),
        ('rating',    pa.int32()),
        ('timestamp', pa.int64()),
    ])

    # Open 16 writers
    writers = {}
    for h in HEX_CHARS:
        p = os.path.join(PARTITION_DIR, f'part_{h}.parquet')
        writers[h] = pq.ParquetWriter(p, part_schema)

    total_kept = 0
    total_read = 0
    buf = []  # accumulate valid records

    print(f'Streaming {os.path.basename(INTERACT_RAW)} ...')
    with gzip.open(INTERACT_RAW, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc='Reading JSONL'):
            total_read += 1
            data = json.loads(line)

            # Filter: rating > 0
            rating = int(data.get('rating', 0))
            if rating == 0:
                continue

            # Filter: book must be in catalog
            book_id = data['book_id']
            if book_id not in valid_books:
                continue

            # Parse date_added -> Unix timestamp
            date_str = data.get('date_added', '')
            try:
                dt = pd.to_datetime(date_str, format='%a %b %d %H:%M:%S %z %Y')
                ts = int(dt.timestamp())
            except Exception:
                continue  # skip unparseable dates

            user_id = data['user_id']
            buf.append({
                'user_id':   user_id,
                'book_id':   book_id,
                'rating':    rating,
                'timestamp': ts,
            })

            # Flush buffer when full
            if len(buf) >= JSONL_CHUNK:
                chunk_df = pd.DataFrame(buf)
                chunk_df['_hb'] = chunk_df['user_id'].str[0].str.lower()

                for bucket, grp in chunk_df.groupby('_hb'):
                    if bucket not in writers:
                        continue
                    sub = grp[['user_id','book_id','rating','timestamp']].copy()
                    sub['rating'] = sub['rating'].astype('int32')
                    sub['timestamp'] = sub['timestamp'].astype('int64')
                    tbl = pa.Table.from_pandas(sub, schema=part_schema, preserve_index=False)
                    writers[bucket].write_table(tbl)

                total_kept += len(chunk_df)
                print(f'  Kept {total_kept:,} / Read {total_read:,}', end='\r')
                buf.clear()
                del chunk_df
                gc.collect()

    # Flush remaining buffer
    if buf:
        chunk_df = pd.DataFrame(buf)
        chunk_df['_hb'] = chunk_df['user_id'].str[0].str.lower()
        for bucket, grp in chunk_df.groupby('_hb'):
            if bucket not in writers:
                continue
            sub = grp[['user_id','book_id','rating','timestamp']].copy()
            sub['rating'] = sub['rating'].astype('int32')
            sub['timestamp'] = sub['timestamp'].astype('int64')
            tbl = pa.Table.from_pandas(sub, schema=part_schema, preserve_index=False)
            writers[bucket].write_table(tbl)
        total_kept += len(chunk_df)
        buf.clear()
        del chunk_df

    # Close all writers
    for w in writers.values():
        w.close()

    del valid_books, writers
    gc.collect()

    print(f'\nTotal lines read: {total_read:,}')
    print(f'Total kept:       {total_kept:,}')
    print_ram('after partitioning')

Loading valid book_ids from cb_books.parquet...
  Valid books: 2,360,648
  [RAM [after loading book ids]] 0.77 GB
Streaming goodreads_interactions_dedup.json.gz ...


Reading JSONL: 2112962it [02:34, 65149.93it/s]

Reading JSONL: 4277136it [05:07, 10001.72it/s]

Reading JSONL: 6430366it [07:40, 9341.54it/s]

Reading JSONL: 8533818it [10:13, 13004.04it/s]

...

Reading JSONL: 224291641it [4:08:13, 14040.87it/s]

Reading JSONL: 226070016it [4:10:38, 15077.67it/s]

Reading JSONL: 227790264it [4:12:59, 12106.78it/s]

Reading JSONL: 228648342it [4:14:18, 14985.23it/s]



Total lines read: 228,648,342
Total kept:       104,551,524
  [RAM [after partitioning]] 1.21 GB


## 3. Filter Valid Users (DuckDB)

Use DuckDB to find users with > 5 interactions. Write valid user list directly to disk via `COPY ... TO`. Then update partition files in-place.

In [ ]:
!pip install -q duckdb

In [ ]:
fimport duckdb

if os.path.exists(VALID_USERS):
    print(f'SKIP: {os.path.basename(VALID_USERS)} already exists.')
else:
    print('Finding valid users via DuckDB...')
    con = duckdb.connect()

    glob_path = os.path.join(PARTITION_DIR, 'part_*.parquet')

    con.execute(f"""
        COPY (
            SELECT user_id
            FROM read_parquet('{glob_path}')
            GROUP BY user_id
            HAVING COUNT(*) > {MIN_INTERACT}
        ) TO '{VALID_USERS}' (FORMAT PARQUET)
    """)

    n_valid = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{VALID_USERS}')"
    ).fetchone()[0]
    print(f'  Valid users (>{MIN_INTERACT} interactions): {n_valid:,}')

    # Update partitions: keep only valid users
    print('Updating partitions to keep valid users only...')
    for h in tqdm(HEX_CHARS, desc='Filtering partitions'):
        pp = os.path.join(PARTITION_DIR, f'part_{h}.parquet')
        tp = os.path.join(PARTITION_DIR, f'part_{h}_tmp.parquet')
        con.execute(f"""
            COPY (
                SELECT p.user_id, p.book_id, p.rating, p.timestamp
                FROM read_parquet('{pp}') p
                SEMI JOIN read_parquet('{VALID_USERS}') v
                    ON p.user_id = v.user_id
            ) TO '{tp}' (FORMAT PARQUET)
        """)
        os.replace(tp, pp)

    con.close()
    print('Partitions updated.')
    print_ram('after DuckDB')

Finding valid users via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Valid users (>5 interactions): 736,390
Updating partitions to keep valid users only...


Filtering partitions:   0%|          | 0/16 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtering partitions:  94%|█████████▍| 15/16 [01:26<00:05,  5.01s/it]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtering partitions: 100%|██████████| 16/16 [01:28<00:00,  5.53s/it]

Partitions updated.
  [RAM [after DuckDB]] 1.17 GB


## 4. Per-User Temporal Train/Test Split

Loop through 16 partition files. For each user:
- Sort interactions by timestamp descending
- Top 20% newest -> Test, rest -> Train
- `math.ceil()` ensures at least 1 test interaction per user

In [ ]:
if os.path.exists(TRAIN_OUT) and os.path.exists(TEST_OUT):
    print('SKIP: Train/Test files already exist.')
    print_ram()
else:
    train_writer = None
    test_writer  = None
    n_train = 0
    n_test  = 0

    for h in tqdm(HEX_CHARS, desc='Splitting train/test'):
        pp = os.path.join(PARTITION_DIR, f'part_{h}.parquet')
        df = pd.read_parquet(pp)

        if df.empty:
            del df
            gc.collect()
            continue

        # Vectorized per-user split
        df = df.sort_values(['user_id', 'timestamp'], ascending=[True, False])
        df['_rank']  = df.groupby('user_id').cumcount()
        df['_count'] = df.groupby('user_id')['user_id'].transform('count')
        df['_ntest'] = np.ceil(df['_count'] * TEST_RATIO).astype(int).clip(lower=1)

        test_mask  = df['_rank'] < df['_ntest']
        train_part = df[~test_mask][['user_id','book_id','rating','timestamp']].copy()
        test_part  = df[test_mask][['user_id','book_id','rating','timestamp']].copy()

        # Write train
        if not train_part.empty:
            t = pa.Table.from_pandas(train_part, preserve_index=False)
            if train_writer is None:
                train_writer = pq.ParquetWriter(TRAIN_OUT, t.schema)
            train_writer.write_table(t)
            n_train += len(train_part)

        # Write test
        if not test_part.empty:
            t = pa.Table.from_pandas(test_part, preserve_index=False)
            if test_writer is None:
                test_writer = pq.ParquetWriter(TEST_OUT, t.schema)
            test_writer.write_table(t)
            n_test += len(test_part)

        del df, train_part, test_part
        gc.collect()

    if train_writer:
        train_writer.close()
    if test_writer:
        test_writer.close()

    print(f'\nTrain: {n_train:,}')
    print(f'Test:  {n_test:,}')
    print(f'Total: {n_train + n_test:,}')
    print_ram('after split')

Splitting train/test: 100%|██████████| 16/16 [03:38<00:00, 13.67s/it]


Train: 83,176,161
Test:  21,166,915
Total: 104,343,076
  [RAM [after split]] 2.29 GB


## 5. Summary

In [ ]:
import pathlib

print('=' * 55)
print('  OUTPUT FILES SUMMARY')
print('=' * 55)

files = [
    ('cb_books.parquet', BOOKS_OUT),
    ('cb_train_interactions.parquet', TRAIN_OUT),
    ('cb_test_interactions.parquet', TEST_OUT),
]
for name, path in files:
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1024**2
        n = pq.read_metadata(path).num_rows
        print(f'  {name:<40} {n:>12,} rows  {sz:>8.1f} MB')
    else:
        print(f'  {name:<40} [NOT FOUND]')

print('=' * 55)
print_ram('final')

  OUTPUT FILES SUMMARY
  cb_books.parquet                            2,360,648 rows    1434.2 MB
  cb_train_interactions.parquet              83,176,161 rows     938.6 MB
  cb_test_interactions.parquet               21,166,915 rows     293.2 MB
  [RAM [final]] 2.29 GB
